# Drugs, Side Effects, and Medical Conditions Analysis

This notebook analyzes a drug dataset containing medical conditions, side effects, drug classes, ratings, reviews, regulatory attributes, pregnancy categories, and alcohol-interaction information.

## Project workflow

- Load and inspect the drug dataset
- Clean numeric and categorical fields
- Handle missing values and duplicates
- Encode categorical variables
- Standardize features
- Explore correlations and frequency distributions
- Extract and analyze side effects, drug classes, and medical conditions
- Apply association-rule mining with Apriori
- Build a decision-tree classifier and evaluate it
- Build an SVM classifier
- Apply DBSCAN and K-Means clustering
- Explore a simple linear-regression analysis

> **Data-use note:** This is a data-analysis and machine-learning project. Its results should not be interpreted as medical advice, treatment guidance, or evidence that a drug is safe or effective for an individual.


## 1. Data Loading and Initial Inspection

In [ ]:
# Import dataset
import pandas as pd
import numpy as np

In [ ]:
# Read the CSV file into a DataFrame
fpath = "../data/drugs_side_effects_drugs_com.csv"
data = pd.read_csv(fpath)

In [ ]:
# Display the columns quantity and names
print('The dataset has {} rows and {} columns'.format(data.shape[0], data.shape[1]))
print("column:")
print(data.columns)

In [ ]:
# Show the main information about dataset
data.info()

In [ ]:
data.head()


In [ ]:
# # Dropping the 'brand_names' column and delete from dataset
# data.drop(columns=['brand_names'], inplace=True)

## 2. Data Cleaning and Missing Values

In [ ]:
# Find duplicate rows based on all columns
duplicate_rows= data[data.duplicated()]
#Count the duplicated rows
duplicate_count = duplicate_rows.shape[0] 
# Print the count of duplicate rows
print("Count of Duplicate Rows:", duplicate_count) 
print(duplicate_rows) # Print the duplicate rows

In [ ]:
# Convert 'rating' and 'no_of_reviews' attributes to numeric
data['rating'] = pd.to_numeric(data['rating'], errors='coerce')
# data['no_of_reviews'] = pd.to_numeric(data['no_of_reviews'], errors='coerce')

print(data.dtypes.value_counts())

In [ ]:
# Convert 'activity' to string, remove whitespace and '%' character, then convert to float and divide by 100
data['activity'] = data['activity'].astype(str).str.replace(r'\s+', '', regex=True).str.rstrip('%').astype('float')/100

# Display the updated 'activity' column
print(data['activity'].head())

In [ ]:
# Print the total number of missing values
print("There are {} missing values in this dataset".format(data.isnull().sum().sum()))
print('Number of instances = %d' % (data.shape[0]))
print('Number of attributes = %d' % (data.shape[1]))
print('Number of missing values:')
for col in data.columns:
    print('\t%s: %d' % (col,data[col].isna().sum()))

In [ ]:
# In the alcohol column we have X and null(NaN) values, because the drug can interact with alcohol or not.
# Therefore, let's replace the values of ak=lcohol column with boolean values.
# Let X will be 1 of interaction, NaN will be 0.
data['alcohol']=data['alcohol'].replace(np.NaN,'0')
data['alcohol']=data['alcohol'].replace({'X': 1})

In [ ]:
# To avoid missing values let's fill them with some information
# In our case we will replace all them
# Fill the null values in 'side_effects' and 'related_drugs' with no
data["side_effects"] = data['side_effects'].fillna('Unknown')
data["related_drugs"] = data['related_drugs'].fillna('Unknown')

In [ ]:
# Fill the null values with 0 as a base for 'rating' and 'no_of_reviews' columns
# It will show that there are no information about it
data["rating"] = data['rating'].fillna('0')
data["no_of_reviews"] = data['no_of_reviews'].fillna('0')

In [ ]:
# Fill the null values with ?
data['generic_name']=data['generic_name'].replace(np.NaN,'Unknown')

# Fill the null values with undefined for 'drug_classes'
data['drug_classes']=data['drug_classes'].replace(np.NaN,'Unknown')

In [ ]:
# For these two columns we already have some category values from dataset's description
# So, let's check the categorical values

# For Rx_OTC
data["rx_otc"].unique()

In [ ]:
# For pregnancy categories
data["pregnancy_category"].unique()

In [ ]:
# Fill the null value with Unknown as a basic value
data['rx_otc']=data['rx_otc'].replace(np.NaN, 'Unknown')

# Fill the null value with Unknown as a basic value
data['pregnancy_category']=data['pregnancy_category'].replace(np.NaN, 'Unknown')

data['no_of_reviews'] = pd.to_numeric(data['no_of_reviews'], errors='coerce')

print(data.head())

dfs=data.copy()

In [ ]:
# Let's check is there any missing values left
print("There are {} missing values in this dataset".format(data.isnull().sum().sum()))
print('Number of instances = %d' % (data.shape[0]))
print('Number of attributes = %d' % (data.shape[1]))
print('Number of missing values:')
for col in data.columns:
    print('\t%s: %d' % (col,data[col].isna().sum()))

In [ ]:
data_version2=data.copy()
print(data_version2.head())
# Print head of dataset to our check

## 3. Save and Reload the Cleaned Dataset

In [ ]:
# Save the data
data_version2.to_csv('../data/drugs_side_effects_drugs_com_version2.csv', index=False)

In [ ]:
# Read the new version dataset
data_ver3=pd.read_csv('../data/drugs_side_effects_drugs_com_version2.csv')

data_ver3["pregnancy_category"].unique()

In [ ]:
data_ver3["csa"].unique()


In [ ]:
data_ver3["rx_otc"].unique()


In [ ]:
data_ver3["generic_name"].unique()


In [ ]:
data_ver3["medical_condition"].unique()


## 4. Categorical Encoding and Feature Preparation

In [ ]:
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
data_ver3["csa"]=label_encoder.fit_transform(data_ver3["csa"])
data_ver3["rx_otc"]=label_encoder.fit_transform(data_ver3["rx_otc"])
data_ver3["generic_name"] = label_encoder.fit_transform(data_ver3["generic_name"])
data_ver3["medical_condition"] = label_encoder.fit_transform(data_ver3["medical_condition"])
data_ver3["pregnancy_category"] = label_encoder.fit_transform(data_ver3["pregnancy_category"])
data_ver3["side_effects"] = label_encoder.fit_transform(data_ver3["side_effects"])

In [ ]:
data_ver3["generic_name"].unique()


In [ ]:
data_ver3["rx_otc"].unique()


In [ ]:
data_ver3["csa"].unique()


In [ ]:
data_ver3["side_effects"].unique()


In [ ]:
data_ver3["medical_condition"].unique()


In [ ]:
data_ver3["pregnancy_category"].unique()


## 5. Standardization and Correlation Analysis

In [ ]:
df=pd.DataFrame(data_ver3,columns=('generic_name', 'medical_condition', 'no_of_reviews', 'side_effects', 'rating', 'csa', 'pregnancy_category', 'rx_otc', 'alcohol'))
df.head(10)

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
scaler.fit(df)
scaled_data=scaler.transform(df)
print(scaled_data)

In [ ]:
df_std = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)
print(df_std)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Heatmap')
plt.show()

## 6. Frequency Analysis

In [ ]:
# Read the new version dataset
data_ver4 = pd.read_csv('../data/drugs_side_effects_drugs_com_version2.csv')

# Importing necessary libraries 
from mlxtend.frequent_patterns import apriori, association_rules
import matplotlib.pyplot as plt
import pandas as pd
# Check for occurrence and frequency of medical conditions, sorted from highest to lowest
medical_condition_counts = data_ver4['medical_condition'].value_counts().sort_values(ascending=False)
print("\nMedical condition occurrence and frequency (sorted from highest to lowest):")
print(medical_condition_counts)

In [ ]:
# Save the results to CSV files if needed
medical_condition_counts.to_csv('../outputs/medical_condition_counts.csv')

In [ ]:
# Importing necessary libraries for processing text
from collections import Counter
import re

# Function to extract side effects from text, split by semicolons
def extract_side_effects(text):
    # Split the text on semicolons then strip whitespace
    return [effect.strip() for effect in re.split(r'[;]', text)]

# Extract and count occurrences of side effects
side_effects = data_ver4['side_effects'].dropna().apply(extract_side_effects).explode()
side_effect_counts = side_effects.value_counts().sort_values(ascending=False)

print("\nSide effects occurrence and frequency (sorted from highest to lowest):")
print(side_effect_counts)


In [ ]:
# Save the side effect counts to a CSV file
side_effect_counts.to_csv('../outputs/side_effect_counts.csv')

In [ ]:
# Function to extract drug classes from text, split by commas
def extract_drug_classes(text):
    # Split the text on commas then strip whitespace
    return [effect.strip() for effect in re.split(r'[,]', text)]

# Extract and count occurrences of drug classes
drug_classes = data_ver4['drug_classes'].dropna().apply(extract_drug_classes).explode()
drug_classes_counts = drug_classes.value_counts().sort_values(ascending=False)

print("\nDrug Classes occurrence and frequency (sorted from highest to lowest):")
print(drug_classes_counts)

## 7. Feature Extraction from Side Effects, Drug Classes, and Conditions

In [ ]:
# Define functions to check for specific side effects and create new boolean columns
def has_hives(text):
    return 'hives' in text.lower()
data_ver4['Hives'] = data_ver4['side_effects'].apply(has_hives)

def has_difficult_breathing(text):
    return 'difficult breathing' in text.lower() or 'difficulty breathing' in text.lower()
data_ver4['Difficult Breathing'] = data_ver4['side_effects'].apply(has_difficult_breathing)

def has_itching(text):
    return 'itching' in text.lower()
data_ver4['Itching'] = data_ver4['side_effects'].apply(has_itching)

In [ ]:
# Define functions to check for specific drug classes and create new boolean columns
def is_usc(text):
    return 'Upper respiratory combinations' in text
data_ver4['Upper respiratory combinations'] = data_ver4['drug_classes'].apply(is_usc)

def is_steriods(text):
    return 'Topical steroids' in text
data_ver4['Topical steroids'] = data_ver4['drug_classes'].apply(is_steriods)

def is_acne(text):
    return 'Topical acne agents' in text
data_ver4['Topical acne agents'] = data_ver4['drug_classes'].apply(is_acne)

In [ ]:
# Define functions to check for specific medical conditions and create new boolean columns
def has_pain(text):
    return 'Pain' in text
data_ver4['Pain'] = data_ver4['medical_condition'].apply(has_pain)

def has_colds_and_flu(text):
    return 'Colds & Flu' in text
data_ver4['Colds & Flu'] = data_ver4['medical_condition'].apply(has_colds_and_flu)

def has_acne(text):
    return 'Acne' in text
data_ver4['Acne'] = data_ver4['medical_condition'].apply(has_acne)

## 8. Exploratory Visualizations

In [ ]:
# Plot the count of occurrences for each side effect
import seaborn as sns

# Plot count of Hives
data_ver4['Hives'].value_counts().plot(kind='bar')
plt.title('Count of Hives')
plt.xlabel('Hives')
plt.ylabel('Count')
plt.xticks([0, 1], ['False', 'True'], rotation=0)
plt.show()

# Plot count of Difficult Breathing
data_ver4['Difficult Breathing'].value_counts().plot(kind='bar')
plt.title('Count of Difficult Breathing')
plt.xlabel('Difficult Breathing')
plt.ylabel('Count')
plt.xticks([0, 1], ['False', 'True'], rotation=0)
plt.show()

# Plot count of Itching
data_ver4['Itching'].value_counts().plot(kind='bar')
plt.title('Count of Itching')
plt.xlabel('Itching')
plt.ylabel('Count')
plt.xticks([0, 1], ['False', 'True'], rotation=0)
plt.show()

In [ ]:
# Plot the count of occurrences for each drug class

# Plot count of Upper respiratory combinations
data_ver4['Upper respiratory combinations'].value_counts().plot(kind='bar')
plt.title('Count of Upper respiratory combinations')
plt.xlabel('Upper respiratory combinations')
plt.ylabel('Count')
plt.xticks([0, 1], ['False', 'True'], rotation=0)
plt.show()

# Plot count of Topical steroids
data_ver4['Topical steroids'].value_counts().plot(kind='bar')
plt.title('Count of Topical steroids')
plt.xlabel('Topical steroids')
plt.ylabel('Count')
plt.xticks([0, 1], ['False', 'True'], rotation=0)
plt.show()

# Plot count of Topical acne agents
data_ver4['Topical acne agents'].value_counts().plot(kind='bar')
plt.title('Count of Topical acne agents')
plt.xlabel('Topical acne agents')
plt.ylabel('Count')
plt.xticks([0, 1], ['False', 'True'], rotation=0)
plt.show()


In [ ]:
# Plot the count of occurrences for each medical condition

# Plot count of Pain
data_ver4['Pain'].value_counts().plot(kind='bar')
plt.title('Count of Pain')
plt.xlabel('Pain')
plt.ylabel('Count')
plt.xticks([0, 1], ['False', 'True'], rotation=0)
plt.show()

# Plot count of Colds & Flu
data_ver4['Colds & Flu'].value_counts().plot(kind='bar')
plt.title('Count of Colds & Flu')
plt.xlabel('Colds & Flu')
plt.ylabel('Count')
plt.xticks([0, 1], ['False', 'True'], rotation=0)
plt.show()

# Plot count of Acne
data_ver4['Acne'].value_counts().plot(kind='bar')
plt.title('Count of Acne')
plt.xlabel('Acne')
plt.ylabel('Count')
plt.xticks([0, 1], ['False', 'True'], rotation=0)
plt.show()


In [ ]:
data_ver5=data_ver4.copy()


## 9. Association Rule Mining with Apriori

In [ ]:
# List of columns those are needed for ARM
columns_to_show = ['Hives', 'Difficult Breathing', 'Itching', 'Upper respiratory combinations', 'Topical steroids', 'Topical acne agents', 'Pain', 'Colds & Flu', 'Acne']

# Create a new DataFrame with only the columns you want to show
data_ver4_subset = data_ver4[columns_to_show]

# Display the first few rows of the subset DataFrame
print("\nFirst few rows of the subset DataFrame:")
print(data_ver4_subset.iloc[900:910])

from mlxtend.frequent_patterns import apriori, association_rules

# Assuming 'data_ver4' is your original DataFrame
columns_to_show = ['Hives', 'Difficult Breathing', 'Itching', 
                   'Upper respiratory combinations', 'Topical steroids', 
                   'Topical acne agents', 'Pain', 'Colds & Flu', 'Acne']

# Create a new DataFrame with only the columns you want to show
data_ver4_subset = data_ver4[columns_to_show]

# Display the first few rows of the subset DataFrame
print("\nFirst few rows of the subset DataFrame:")
print(data_ver4_subset.iloc[900:910])


In [ ]:
# Convert columns to boolean
bool_columns = ['Hives', 'Difficult Breathing', 'Itching', 
                'Upper respiratory combinations', 'Topical steroids', 
                'Topical acne agents', 'Pain', 'Colds & Flu', 'Acne']
for col in bool_columns:
    data_ver4[col] = data_ver4[col].astype(bool)
    
# Create new columns for combined drug classes and medical conditions with 'T' and 'F'
data_ver4['DrugClass_MedCondition'] = (
    data_ver4['Upper respiratory combinations'].replace({True: 'T', False: 'F'}) + 
    data_ver4['Topical steroids'].replace({True: 'T', False: 'F'}) + 
    data_ver4['Topical acne agents'].replace({True: 'T', False: 'F'}) + 
    data_ver4['Pain'].replace({True: 'T', False: 'F'}) +
    data_ver4['Colds & Flu'].replace({True: 'T', False: 'F'}) +
    data_ver4['Acne'].replace({True: 'T', False: 'F'})
)
# Verify the creation of the new column
print("\nFirst few rows of the DataFrame with the new combined column:")
print(data_ver4[['DrugClass_MedCondition']].head())

In [ ]:
# Filter out rows where DrugClass_MedCondition has fewer than 2 'T's
data_ver4 = data_ver4[data_ver4['DrugClass_MedCondition'].str.count('T') >= 2]

# Verify the creation of the new column
print("\nFirst few rows of the DataFrame with the new combined column with at least two T's:")
print(data_ver4[['DrugClass_MedCondition']].head())

In [ ]:
# One-hot encode the new combined column and the side effects
data_ver4_subset = pd.get_dummies(data_ver4[['DrugClass_MedCondition', 'Hives', 'Difficult Breathing', 'Itching']])
data_ver4_subset.head()

In [ ]:
# Apply the Apriori algorithm
freq_items = apriori(data_ver4_subset, min_support=0.2, use_colnames=True, verbose=1)

# Generate the association rules
rules = association_rules(freq_items, metric="confidence", min_threshold=0.5)

# Filter rules to ensure DrugClass_MedCondition is in antecedents and side effects are in consequents
side_effects = {'Hives', 'Difficult Breathing', 'Itching'}
rules_filtered = rules[
    rules['antecedents'].apply(lambda x: any(item.startswith('DrugClass_MedCondition_') for item in x) and not any(item in side_effects for item in x)) &
    rules['consequents'].apply(lambda x: any(item in side_effects for item in x) and len(x) == 1)
    ]

print(rules_filtered.head(30))

In [ ]:
# Ignore deprecation warnings for seaborn
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Create the scatter plot for support vs confidence
plt.figure(figsize=(10, 5))
sns.scatterplot(x='support', y='confidence', size='lift', data=rules, legend=False, sizes=(20, 2000))
plt.title('Support vs Confidence')
plt.xlabel('Support')
plt.ylabel('Confidence')
plt.show()

In [ ]:
# Define the function to create the 'Drug Class' column
def classify_drug(row):
    if row['Upper respiratory combinations'] == 1:
        return 'URC'
    else:
        return 'Non-URC'

# Apply the function to create the new 'Drug Class' column
data_ver5['Drug Class'] = data_ver5.apply(classify_drug, axis=1)

# Print a subset of the DataFrame to verify the results
print(data_ver5.iloc[900:910])

In [ ]:
# List of columns those are needed for ARM
new_columns = ['drug_name','Hives', 'Difficult Breathing', 'Itching', 'Pain', 'Colds & Flu', 'Acne',  'Drug Class']

# Create a new DataFrame with only the columns you want to show
data_ver5_subset = data_ver5[new_columns]

# Display the first few rows of the subset DataFrame
print("\nFirst few rows of the subset DataFrame:")
print(data_ver5_subset.iloc[900:910])

In [ ]:
print(data_ver5_subset.columns)


## 10. Decision Tree Classification

In [ ]:
from sklearn import tree
import matplotlib.pyplot as plt

Y = data_ver5_subset['Drug Class']
X = data_ver5_subset.drop(['drug_name', 'Drug Class'],axis=1)

clf = tree.DecisionTreeClassifier(criterion='entropy', max_depth=3)
clf = clf.fit(X, Y)

In [ ]:
plt.figure(figsize=(20, 10))
tree.plot_tree(clf, feature_names=['Hives', 'Difficult Breathing', 'Itching', 'Pain', 'Colds & Flu', 'Acne'], class_names=['URC', 'Non-URC'], filled=True, rounded=True, fontsize=12)
plt.show()


In [ ]:
from sklearn.tree import export_text
tree_rules = export_text(clf, feature_names=['Hives', 'Difficult Breathing', 'Itching', 'Pain', 'Colds & Flu', 'Acne'])
print(tree_rules)

## 11. Decision Tree Evaluation

In [ ]:
# split the dataset
from sklearn.model_selection import train_test_split
X_train, test_x, y_train, test_lab = train_test_split(X,Y,test_size = 0.4, random_state = 42)
clf = clf.fit(X_train, y_train)
test_pred_decision_tree = clf.predict(test_x)

In [ ]:
# import the relevant packages
from sklearn import metrics
# get the confusion matrix
confusion_matrix = metrics.confusion_matrix(test_lab, test_pred_decision_tree)

# turn this into a dataframe
matrix_df = pd.DataFrame(confusion_matrix)
# plot the result
ax = plt.axes()
sns.set(font_scale=1.3)
plt.figure(figsize=(15,10))
sns.heatmap(matrix_df, annot=True, fmt='g', ax=ax, cmap='magma')
# set axis titles
ax.set_title('Confusion Matrix - Decision Tree')
ax.set_xlabel("Predicted label", fontsize =15)
ax.set_xticklabels(['URC', 'Non-URC'])
ax.set_ylabel('True label', fontsize =15)
ax.set_yticklabels(['URC', 'Non-URC'], rotation = 0)
plt.show()

In [ ]:
print(metrics.classification_report(test_lab, test_pred_decision_tree))


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt


# Check lengths of X and Y
if len(data_ver5_subset['Drug Class']) == 2498:
    data = data_ver5_subset.iloc[:2498]  # Trim the data to match the shorter length

# Assign features and target variable
Y = data_ver5_subset['Drug Class']
X = data_ver5_subset[['Hives', 'Difficult Breathing', 'Itching', 'Pain', 'Colds & Flu', 'Acne']]

# Ensure X and Y have the same length
assert len(X) == len(Y)

# Training and Test set creation
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=1)

# Model fitting and evaluation
maxdepths = [2, 3, 4, 5, 6, 7, 8, 9, 10, 15, 20, 25, 30, 35, 40, 45, 50]
trainAcc = np.zeros(len(maxdepths))
testAcc = np.zeros(len(maxdepths))

index = 0
for depth in maxdepths:
    clf = tree.DecisionTreeClassifier(max_depth=depth)
    clf = clf.fit(X_train, Y_train)
    Y_predTrain = clf.predict(X_train)
    Y_predTest = clf.predict(X_test)
    trainAcc[index] = accuracy_score(Y_train, Y_predTrain)
    testAcc[index] = accuracy_score(Y_test, Y_predTest)
    index += 1

## 12. Support Vector Machine Classification

In [ ]:
# import svm model
from sklearn import svm

# split the dataset
from sklearn.model_selection import train_test_split
X_train, test_x, y_train, test_lab = train_test_split(X,Y, test_size= 0.2, random_state =1)

# Create a svm Classifier
clf = svm.SVC(kernel='linear') # Linear Kernel

# Train the model using the training sets
clf.fit(X_train, y_train)

# Predict the response for test dataset
y_pred = clf.predict(test_x)


# Import scikit-learn metrics module for accuracy calculation
from sklearn import metrics

# Model Accuracy: how often is the classifier correct?
print("Accuracy:", metrics.accuracy_score(Y_test, Y_predTest))

# Model Precision: what percentage of positive tuples are labeled as such?
print("Precision:", metrics.precision_score(Y_test, Y_predTest, average='macro'))

# Model Recall: what percentage of positive tuples are labelled as such?
print("Recall:", metrics.recall_score(Y_test, Y_predTest, average='macro'))

# Model F1 Score: weighted average of precision and recall
print("F1 Score:", metrics.f1_score(Y_test, Y_predTest, average='macro'))

In [ ]:
data_ver6 = df
data_ver6

## 13. Clustering Analysis

In [ ]:
# Drop rows where 'rating' and 'no_of_reviews' are 0
data_ver6 = data_ver6[(data_ver6['rating'] != 0) & (data_ver6['no_of_reviews'] != 0)]

# Define a function to categorize the ratings
def categorize_rating(rating):
    if rating <= 5.0:
        return '0.0 to 5.0'
    else:
        return '5.1 to 10.0'

# Apply the function to create a new column 'rating_category'
data_ver6['rating_category'] = data_ver6['rating'].apply(categorize_rating)

# Filter rows with ratings 5.0 to 10.0
filtered_data = data_ver6[data_ver6['rating'] > 5.0]

# # Display the updated DataFrame
# print(data_ver6.head())

# # Display rows 500 to 515
# print(data_ver6.iloc[500:515])

# Display the number of rows and columns
num_rows, num_columns = data_ver6.shape
print(f'Number of rows: {num_rows}')
print(f'Number of columns: {num_columns}')

# Sort the DataFrame by 'no_of_reviews' from highest to lowest
sorted_data = data_ver6.sort_values(by='no_of_reviews', ascending=False)

# Display the sorted DataFrame
print(sorted_data)

In [ ]:
# # """No of Reviews and Side Effects KMeans"""

# from sklearn.preprocessing import StandardScaler
# from sklearn.cluster import KMeans

# # Feature selection for clustering
# features = filtered_data[['no_of_reviews', 'side_effects', 'csa', 'pregnancy_category', 'rx_otc', 'alcohol']]

# # # Standardize the features
# scaler = StandardScaler()
# scaled_features = scaler.fit_transform(features)
# # Apply K-means clustering
# kmeans = KMeans(n_clusters=2, random_state=42)
# clusters = kmeans.fit_predict(scaled_features)

# # Add cluster labels to the DataFrame
# filtered_data['cluster'] = clusters

# # Display the DataFrame with cluster labels
# print(filtered_data.head())

# # Visualize the clusters (example using no_of_reviews and side_effects)
# plt.scatter(filtered_data['no_of_reviews'], filtered_data['side_effects'], c=filtered_data['cluster'], cmap='viridis')
# plt.xlabel('No of Reviews')
# plt.ylabel('Side Effects')
# plt.title('KMeans Clusters based on No of Reviews and Side Effects')
# plt.show()

In [ ]:
"""No of Reviews and Side Effects DBSCAN"""

from sklearn.cluster import DBSCAN

# Feature selection for clustering
features = filtered_data[['no_of_reviews', 'side_effects', 'csa', 'pregnancy_category', 'rx_otc', 'alcohol']]

# Standardize the features
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

# Apply DBSCAN clustering
dbscan = DBSCAN(eps=0.5, min_samples=5)
clusters = dbscan.fit_predict(scaled_features)

# Add cluster labels to the DataFrame
filtered_data['cluster'] = clusters

# Display the DataFrame with cluster labels
print(filtered_data.head())

# Visualize the clusters (example using no_of_reviews and side_effects)
plt.scatter(filtered_data['no_of_reviews'], filtered_data['side_effects'], c=filtered_data['cluster'], cmap='viridis')
plt.xlabel('No of Reviews')
plt.ylabel('Side Effects')
plt.title('DBSCAN Clusters based on No of Reviews and Side Effects')
plt.show()

In [ ]:
# # # """Number of reviews with rating > 5.0 KMeans"""

# from sklearn.preprocessing import StandardScaler
# from sklearn.cluster import KMeans

# # # Filter rows with ratings 5.0 to 10.0
# filtered_data = data_ver6[data_ver6['rating'] > 5.0]

# # # Select the 'no_of_reviews' column for clustering
# features = filtered_data[['no_of_reviews']]

# # # Standardize the features
# scaler = StandardScaler()
# scaled_features = scaler.fit_transform(features)

# # # Apply K-means clustering
# kmeans = KMeans(n_clusters=2, random_state=42)
# clusters = kmeans.fit_predict(scaled_features)

# # # Add cluster labels to the DataFrame
# filtered_data['kmeans_cluster'] = clusters

# # # Display the DataFrame with cluster labels
# print(filtered_data.head())

# # # Visualize the clusters
# plt.scatter(filtered_data['no_of_reviews'], filtered_data['rating'], c=filtered_data['kmeans_cluster'], cmap='viridis')
# plt.xlabel('No of Reviews')
# plt.ylabel('Rating')
# plt.title('K-means Clusters based on No of Reviews and Rating')
# plt.show()

In [ ]:
data_ver7 = data_ver5_subset.copy()
data_ver7.info()

In [ ]:
# Create a copy of the DataFrame
# data_ver7 = data_ver5.copy()

# Convert boolean columns to integers (0 and 1)
boolean_columns = ['Hives', 'Difficult Breathing', 'Itching', 'Pain', 'Colds & Flu', 'Acne', 'Drug Class']

# For 'Drug Class', encode 'URC' as 1 and 'Non-URC' as 0
data_ver7['Drug Class'] = data_ver7['Drug Class'].map({'URC': 1, 'Non-URC': 0})

# Convert other boolean columns to integers
for column in boolean_columns[1:]:
    data_ver7[column] = data_ver7[column].astype(int)

# Select only the boolean columns converted to numerical values
numerical_columns = data_ver7[boolean_columns]

# Calculate the correlation matrix
correlation_matrix = numerical_columns.corr()

# # Create the heatmap
# sns.heatmap(correlation_matrix, annot=True, cmap="YlGnBu", cbar=True)
# plt.show()

# Create a high-resolution heatmap
plt.figure(figsize=(20, 10), dpi=300) # Increase figure size and DPI for higher resolution
sns.heatmap(correlation_matrix, annot=True, cmap="YlGnBu", cbar=True, annot_kws={'size':20})
plt.title('Correlation Heatmap of Boolean Columns')
plt.show()

## 14. Regression Analysis

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score


# Encode 'Drug Class' and 'Colds & Flu'
data_ver7['Drug Class'] = label_encoder.fit_transform(data_ver5['Drug Class'])
data_ver7['Colds & Flu'] = data_ver7['Colds & Flu'].astype(int)

# Prepare features and target variable
X = data_ver7[['Drug Class']]
y = data_ver7['Colds & Flu']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and train the model
model = LinearRegression()
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Calculate performance metrics
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse}")
print(f"R-squared: {r2}")

# Print the model coefficients
print(f"Coefficient: {model.coef_[0]}")
print(f"Intercept: {model.intercept_}")

## 15. K-Means Clustering of Ratings and Reviews

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# Extract relevant columns
clustering_data = data_ver6[['rating', 'no_of_reviews']].dropna()

# Perform K-means clustering
kmeans = KMeans(n_clusters=3, random_state=0).fit(clustering_data)

# Add cluster labels to the original data
clustering_data['cluster'] = kmeans.labels_

# Plot the clusters
plt.figure(figsize=(10, 6))
plt.scatter(clustering_data['rating'], clustering_data['no_of_reviews'], c=clustering_data['cluster'], cmap='viridis')
plt.xlabel('Rating')
plt.ylabel('Number of Reviews')
plt.title('K-means Clustering of Drugs Based on Rating and Number of Reviews')
plt.colorbar(label='Cluster')
plt.show()


In [ ]:
# Cluster 0 (purple): Concentrated at the lower end of both ratings and reviews.
# Cluster 1 (green): Spread across a broader range of ratings and reviews.
# Cluster 2 (yellow): Includes entries with relatively higher ratings and a large number of reviews.